In [3]:
#r "nuget: Plotly.NET, 5.0.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.13.0"
#r "nuget: MathNet.Numerics, 5.0.0"

using System;
using System.Collections.Generic;
using System.Net.Http;
using System.IO;
using System.Linq;
using System.Globalization;
using Plotly.NET;
using Plotly.NET.LayoutObjects;
using Plotly.NET.Interactive;
using Plotly.NET.CSharp;
using Chart = Plotly.NET.CSharp.Chart;
using MathNet.Numerics.LinearAlgebra;
using MathNet.Numerics.LinearAlgebra.Double;
using System.Text.RegularExpressions;

public class Track
{
    public string Genre { get; set; }
    public string ArtistName { get; set; }
    public string TrackName { get; set; }
    public string TrackId { get; set; }
    public double Popularity { get; set; }
    
    public double[] Features { get; set; } 
    public double[] StdFeatures { get; set; } 
    public int Cluster { get; set; }
    public double[] PCA { get; set; } 
}

string filePath = "SpotifyFeatures.csv";
var lines = File.ReadAllLines(filePath).Skip(1).ToList();

var allTracks = new List<Track>();
var regex = new Regex(",(?=(?:[^\"]*\"[^\"]*\")*[^\"]*$)");

foreach (var line in lines)
{
    var cols = regex.Split(line);
    if (cols.Length < 18) continue;

    try
    {
        var track = new Track
        {
            Genre = cols[0],
            ArtistName = cols[1],
            TrackName = cols[2],
            TrackId = cols[3],
            Popularity = double.Parse(cols[4], CultureInfo.InvariantCulture),
            Features = new double[]
            {
                double.Parse(cols[5], CultureInfo.InvariantCulture), 
                double.Parse(cols[6], CultureInfo.InvariantCulture), 
                double.Parse(cols[7], CultureInfo.InvariantCulture), 
                double.Parse(cols[8], CultureInfo.InvariantCulture),
                double.Parse(cols[9], CultureInfo.InvariantCulture), 
                double.Parse(cols[11], CultureInfo.InvariantCulture),
                double.Parse(cols[12], CultureInfo.InvariantCulture), 
                double.Parse(cols[14], CultureInfo.InvariantCulture), 
                double.Parse(cols[15], CultureInfo.InvariantCulture), 
                double.Parse(cols[17], CultureInfo.InvariantCulture) 
            }
        };
        allTracks.Add(track);
    }
    catch {}
}
int featureCount = allTracks[0].Features.Length;
double[] means = new double[featureCount];
double[] stdDevs = new double[featureCount];

for (int i = 0; i < featureCount; i++)
{
    means[i] = allTracks.Average(t => t.Features[i]);
    double variance = allTracks.Average(t => Math.Pow(t.Features[i] - means[i], 2));
    stdDevs[i] = Math.Sqrt(variance);
}

foreach (var track in allTracks)
{
    track.StdFeatures = new double[featureCount];
    for (int i = 0; i < featureCount; i++)
    {
        track.StdFeatures[i] = stdDevs[i] == 0 ? 0 : (track.Features[i] - means[i]) / stdDevs[i];
    }
}
var popularTracks = allTracks.Where(t => t.Popularity >= 85).ToList();
Console.WriteLine($"Всього треків: {allTracks.Count}");
Console.WriteLine($"Треків з Popularity >= 85: {popularTracks.Count}");



Installed Packages MathNet.Numerics, 5.0.0 Plotly.NET, 5.0.0 Plotly.NET.CSharp, 0.13.0 Plotly.NET.Interactive, 5.0.0

Всього треків: 232725
Треків з Popularity >= 85: 417
